# Estimating accuracy with stratified k-fold cross-validation

Because `LanguageModelClassifier` honours the estimator contract, it drops straight into
`cross_val_score`. Each fold gets a freshly `clone`-d estimator — so no fitted state leaks between
folds — and is fine-tuned from the base model independently.

## Why stratified, and what the numbers mean

$k$-fold CV partitions the data into $k$ folds, trains on $k-1$ of them and tests on the held-out
fold, $k$ times, yielding scores $s_1,\dots,s_k$. The reported estimate is their mean with its
spread,

$$ \bar{s} = \frac{1}{k}\sum_{i=1}^{k} s_i, \qquad
\hat{\sigma} = \sqrt{\frac{1}{k}\sum_{i=1}^{k} (s_i - \bar{s})^2}. $$

**Stratified** folds keep each fold's class proportions close to the full dataset's. On small or
imbalanced data a plain random split can under- or over-represent a class in a fold — or drop it
entirely — which inflates variance and biases the estimate; stratification is the right default for
classification.

We also print an interval from the empirical 2.5th and 97.5th percentiles of the fold scores. With
only $k = 10$ folds this is a coarse summary of spread, not a rigorous confidence interval, but it
conveys how much the score moves across splits. distilgpt2 on Iris is a weak classifier; the point
here is the **evaluation protocol**, not the score.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklm import GenerationConfig, LanguageModelClassifier, LoggingCallback, TrainingConfig

sns.set_theme(style="whitegrid", context="notebook")
SEED = 42

## Data and estimator

`LoggingCallback(log_every="epoch")` keeps the per-fold output quiet — a live UI callback would
redraw once per fold.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data.rename(columns=lambda c: c.removesuffix(" (cm)"))
y = iris.target_names[iris.target]

clf = LanguageModelClassifier(
    backend="mlx",
    model="gabfssilva/distilgpt2",
    training=TrainingConfig(
        epochs=3, batch_size=16, lr_scheduler="constant", augmentation_factor=24
    ),
    generation=GenerationConfig(n_samples=24),
    callback=LoggingCallback(log_every="epoch"),
    random_state=SEED,
)

## Run cross-validation

This fits the model ten times (once per fold), so expect it to take a while.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")

print("per-fold accuracy: " + ", ".join(f"{s:.3f}" for s in scores))
print(f"mean +/- std:      {scores.mean():.3f} +/- {scores.std():.3f}")
lo, hi = np.percentile(scores, 2.5), np.percentile(scores, 97.5)
print(f"95% interval:      [{lo:.3f}, {hi:.3f}]")

## Spread across folds

The box shows the interquartile range and median; the dots are the individual fold scores. A tight
cluster means the estimate is stable across splits.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x=scores, ax=ax, color="#378add", width=0.4)
sns.stripplot(x=scores, ax=ax, color="black", size=7, alpha=0.7)
ax.set_xlabel("fold accuracy")
ax.set_title("Per-fold accuracy (10-fold stratified CV)")
plt.tight_layout()
plt.show()